In [1]:
from dotenv import load_dotenv
from pprint import pprint
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain.messages import HumanMessage

load_dotenv()

True

In [ ]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [2]:
client: MultiServerMCPClient = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": "python",
            "args": ["resources/2.1_mcp_server.py"]
        }
    }
)

In [3]:
# get tools
tools = await client.get_tools()
pprint(tools)

[StructuredTool(name='search_web', description='Search the web for information', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_webArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x1162fd260>)]


In [4]:
# get resources
resources = await client.get_resources("local_server")
pprint(resources)

[Blob 4667749824]


In [5]:
# get prompts
prompts = await client.get_prompt("local_server", "prompt")
prompt = prompts[0].content
pprint(prompt)

('\n'
 '    You are a helpful assistant that answers user questions about LangChain, '
 'LangGraph and LangSmith.\n'
 '\n'
 '    You can use the following tools/resources to answer user questions:\n'
 '    - search_web: Search the web for information\n'
 '    - github_file: Access the langchain-ai repo files\n'
 '\n'
 '    If the user asks a question that is not related to LangChain, LangGraph '
 'or LangSmith, you should say "I\'m sorry, I can only answer questions about '
 'LangChain, LangGraph and LangSmith."\n'
 '\n'
 "    You may try multiple tool and resource calls to answer the user's "
 'question.\n'
 '\n'
 '    You may also ask clarifying questions to the user to better understand '
 'their question.\n'
 '    ')


In [6]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [7]:
config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    input={"messages": [HumanMessage(content="Tell me about redis")]},
    config=config
)

In [8]:
pprint(response)

{'messages': [HumanMessage(content='Tell me about redis', additional_kwargs={}, response_metadata={}, id='9bb89d3b-dda1-42eb-a267-cdac5b80e211'),
              AIMessage(content='I’m sorry, I can only answer questions about LangChain, LangGraph, and LangSmith.\n\nIf you meant Redis in the context of LangChain, I can help with topics like:\n- Using Redis as a vector store (e.g., storing and querying embeddings) for retrieval-augmented generation.\n- Using Redis as a cache layer to speed up repeated LLM calls or store intermediate results.\n- Storing conversation history or memory/state in Redis for multi-session or multi-user apps.\n- Connecting LangChain components to Redis-backed services or state stores.\n\nWhich Redis-related use case are you interested in within LangChain? If you can share a bit more (e.g., “vector store with Redis,” “caching LLM calls in Redis,” or “Redis as memory for an agent”), I can provide a more precise explanation and example code or point you to the exact 

In [9]:
response = await agent.ainvoke(
    input={"messages": [HumanMessage(content="Tell me about langchain")]},
    config=config
)
pprint(response)

{'messages': [HumanMessage(content='Tell me about langchain', additional_kwargs={}, response_metadata={}, id='6e4d94a3-eca6-401b-891b-23b1b239b69a'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 477, 'prompt_tokens': 265, 'total_tokens': 742, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DW9rZKcYkwephnwdOj3lAdjOTZjRE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da309-3004-72e0-8c36-c61423d7dda5-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'LangChain LangSmith LangGraph overview'}, 'id': 'call_rVQ0rKJ4NWjxUNWiB0P8Ie4Y', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metad

## Online MCP

In [13]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()
pprint(tools)

[StructuredTool(name='get_current_time', description='Get current time in a specific timezones', args_schema={'type': 'object', 'properties': {'timezone': {'type': 'string', 'description': "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no timezone provided by the user."}}, 'required': ['timezone']}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x11666c4a0>),
 StructuredTool(name='convert_time', description='Convert time between timezones', args_schema={'type': 'object', 'properties': {'source_timezone': {'type': 'string', 'description': "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no source timezone provided by the user."}, 'time': {'type': 'string', 'description': 'Time to convert in 24-hour format (HH:MM)'}, 'target_timezone': {'type': 'string', 'description': "Target IANA timezone 

In [15]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [16]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    input={"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='9e4ffe07-36cb-4ae1-9221-613d641e5f4a'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 296, 'total_tokens': 451, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DW8kicqJdSVXpSsdArJ4jLjJQCIhy', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da2c8-07ad-7191-bab7-4edb14b67944-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'call_l9Fu2gNxj65wAjZzyDAnRKE3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens':